# S06 Phase A — three-way model evaluation

Runs BOFBL, RRTN, DeepRemaster on 50 stratified cat_a frames from Lanka Dahan (1917).

**Runtime:** Colab with T4 GPU (Runtime → Change runtime type → T4 GPU).

**Flow:** Section 0 sets up Drive + inputs. Sections 1/2/3 run each model. **Restart the runtime between sections** to avoid dependency collisions.

**Outputs** go to `/MyDrive/silent-film-restoration/lanka-dahan/s06-eval/output/<model>/`. Pull them back locally and run `build_comparison.py`.

## 0. Setup — Drive, paths, GPU check (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, subprocess, time, shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/silent-film-restoration/lanka-dahan/s06-eval')
INPUT_DIR   = DRIVE_ROOT / 'input'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
OUTPUT_DIR  = DRIVE_ROOT / 'output'
BENCH_DIR   = DRIVE_ROOT / 'bench'
for d in (INPUT_DIR, WEIGHTS_DIR, OUTPUT_DIR, BENCH_DIR):
    d.mkdir(parents=True, exist_ok=True)

# First-run: pull the 50-frame input tarball from the GitHub release and
# stage it on Drive. Idempotent — skips if already present. Filters
# AppleDouble '._*' sidecars that older tarballs may have carried.
RELEASE_URL = 'https://github.com/utsavbansal93/Silent-Film-Restoration/releases/download/s06-eval-input/s06-eval-input.tar.gz'
for stale in list(INPUT_DIR.glob('._*')):
    stale.unlink()
n_inputs = len([p for p in INPUT_DIR.glob('*.png') if not p.name.startswith('._')])
if n_inputs != 50:
    print(f'fetching inputs ({n_inputs}/50 present)...')
    shutil.rmtree('/tmp/input', ignore_errors=True)
    !curl -sL {RELEASE_URL} -o /tmp/in.tgz && tar -xzf /tmp/in.tgz -C /tmp/ --exclude='._*'
    for p in Path('/tmp/input').glob('*.png'):
        if p.name.startswith('._'):
            continue
        shutil.copy(p, INPUT_DIR / p.name)
    n_inputs = len([p for p in INPUT_DIR.glob('*.png') if not p.name.startswith('._')])
print(f'inputs on Drive: {n_inputs}')
assert n_inputs == 50, f'expected 50 input PNGs in {INPUT_DIR}, found {n_inputs}'

In [ ]:
!nvidia-smi | head -15

---
## 1. BOFBL — Bringing Old Films Back to Life (CVPR 2022)

Repo: https://github.com/raywzy/Bringing-Old-Films-Back-to-Life

**Known quirks:** expects RGB input even for B&W; flow module can fail on short sequences — handle one frame at a time treating each as a 1-frame sequence, or replicate to a 3-frame window.

In [ ]:
%cd /content
!rm -rf /content/bofbl && git clone https://github.com/raywzy/Bringing-Old-Films-Back-to-Life.git /content/bofbl
%cd /content/bofbl
# Pin to a known-good commit if master drifts. Leave free-running for now.

In [ ]:
# Weights: check Drive cache first. BOFBL's README points at a Google Drive link
# for `Label_Track-net.pth`, `Mask_CRSC_net.pth`, `Restoration_net.pth`. On first
# run, download manually and drop them into WEIGHTS_DIR/bofbl/.
BOFBL_WEIGHTS = WEIGHTS_DIR / 'bofbl'
BOFBL_WEIGHTS.mkdir(parents=True, exist_ok=True)
print('weights present:', sorted(p.name for p in BOFBL_WEIGHTS.glob('*.pth')))
# Symlink into the repo's checkpoint dir so test scripts find them.
repo_ckpt = Path('/content/bofbl/checkpoints')
repo_ckpt.mkdir(exist_ok=True)
for pth in BOFBL_WEIGHTS.glob('*.pth'):
    dst = repo_ckpt / pth.name
    if not dst.exists():
        dst.symlink_to(pth)

In [ ]:
# Install BOFBL's deps. Its requirements.txt is sometimes stale; using Colab's
# pre-installed torch/torchvision generally works. Override only what's needed.
!pip install -q einops==0.6.1 scikit-image==0.21.0 opencv-python-headless==4.8.1.78

In [ ]:
# Inference. BOFBL's entry script is test.py at repo root. It expects a folder
# of frames and writes to an output folder. See repo README for exact flags.
BOFBL_OUT = OUTPUT_DIR / 'bofbl'
BOFBL_OUT.mkdir(parents=True, exist_ok=True)
BOFBL_BENCH = BENCH_DIR / 'bofbl.json'

t0 = time.time()
# NOTE: this exact invocation may need adjustment once we see BOFBL's current
# test.py signature. Treat as a placeholder to confirm/edit on first run.
cmd = [
    'python', '/content/bofbl/test.py',
    '--test_path', str(INPUT_DIR),
    '--output_dir', str(BOFBL_OUT),
    '--gpu_ids', '0',
]
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd='/content/bofbl', capture_output=True, text=True)
wall = time.time() - t0
print(proc.stdout[-2000:])
print('STDERR:', proc.stderr[-2000:])
n_out = len(list(BOFBL_OUT.glob('*.png')))
BOFBL_BENCH.write_text(json.dumps({
    'model': 'bofbl',
    'wall_s': round(wall, 1),
    'n_frames': n_out,
    'fps': round(n_out / max(wall, 1e-6), 2),
    'returncode': proc.returncode,
    'cmd': cmd,
}, indent=2))
print(f'BOFBL: {n_out} frames, {wall:.1f}s')

---
## 2. RRTN — Recursive Recurrent Transformer (builds on BOFBL)

Repo: https://github.com/mountln/RRTN-old-film-restoration

**Restart runtime before running this section** (Runtime → Restart runtime), then re-run cells 0.1–0.3 above before continuing.

In [ ]:
%cd /content
!rm -rf /content/rrtn && git clone https://github.com/mountln/RRTN-old-film-restoration.git /content/rrtn
%cd /content/rrtn

In [ ]:
RRTN_WEIGHTS = WEIGHTS_DIR / 'rrtn'
RRTN_WEIGHTS.mkdir(parents=True, exist_ok=True)
print('weights present:', sorted(p.name for p in RRTN_WEIGHTS.glob('*.pth')))
# Symlink into repo checkpoints.
repo_ckpt = Path('/content/rrtn/checkpoints')
repo_ckpt.mkdir(exist_ok=True)
for pth in RRTN_WEIGHTS.glob('*.pth'):
    dst = repo_ckpt / pth.name
    if not dst.exists():
        dst.symlink_to(pth)

In [ ]:
!pip install -q -r /content/rrtn/requirements.txt 2>&1 | tail -5

In [ ]:
RRTN_OUT = OUTPUT_DIR / 'rrtn'
RRTN_OUT.mkdir(parents=True, exist_ok=True)
RRTN_BENCH = BENCH_DIR / 'rrtn.json'

t0 = time.time()
# Placeholder invocation — RRTN's test.py takes --input and --output.
cmd = [
    'python', '/content/rrtn/test.py',
    '--input', str(INPUT_DIR),
    '--output', str(RRTN_OUT),
]
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd='/content/rrtn', capture_output=True, text=True)
wall = time.time() - t0
print(proc.stdout[-2000:])
print('STDERR:', proc.stderr[-2000:])
n_out = len(list(RRTN_OUT.glob('*.png')))
RRTN_BENCH.write_text(json.dumps({
    'model': 'rrtn',
    'wall_s': round(wall, 1),
    'n_frames': n_out,
    'fps': round(n_out / max(wall, 1e-6), 2),
    'returncode': proc.returncode,
    'cmd': cmd,
}, indent=2))
print(f'RRTN: {n_out} frames, {wall:.1f}s')

---
## 3. DeepRemaster — Iizuka et al. SIGGRAPH Asia 2019

Repo: https://github.com/satoshiiizuka/siggraphasia2019_remastering

**Restart runtime before running this section**, then re-run cells 0.1–0.3 above.

**Known quirks:** DeepRemaster operates on sequences (temporal context). For 50 independent still frames, feeding a pseudo-sequence of length 1 may degrade output quality — consider running in small clips and accepting per-frame output.

In [ ]:
%cd /content
!rm -rf /content/deepremaster && git clone https://github.com/satoshiiizuka/siggraphasia2019_remastering.git /content/deepremaster
%cd /content/deepremaster

In [ ]:
DR_WEIGHTS = WEIGHTS_DIR / 'deepremaster'
DR_WEIGHTS.mkdir(parents=True, exist_ok=True)
# DeepRemaster ships a download script; cache to Drive.
if not any(DR_WEIGHTS.glob('*.pth')):
    !cd /content/deepremaster && bash download_model.sh
    for pth in Path('/content/deepremaster/model').glob('*.pth'):
        shutil.copy(pth, DR_WEIGHTS / pth.name)
print('weights present:', sorted(p.name for p in DR_WEIGHTS.glob('*.pth')))
# Mirror back so repo's own path works.
repo_model = Path('/content/deepremaster/model')
repo_model.mkdir(exist_ok=True)
for pth in DR_WEIGHTS.glob('*.pth'):
    dst = repo_model / pth.name
    if not dst.exists():
        shutil.copy(pth, dst)

In [ ]:
!pip install -q opencv-python-headless==4.8.1.78 scipy==1.11.4

In [ ]:
# DeepRemaster expects a video file. Synthesize one from the 50 PNGs, run the
# model, then extract output frames back to individual PNGs.
WORK = Path('/content/dr_work'); WORK.mkdir(exist_ok=True)
frames = sorted(INPUT_DIR.glob('*.png'))
# Numeric symlinks for ffmpeg.
dr_in = WORK / 'in'; dr_in.mkdir(exist_ok=True)
for i, f in enumerate(frames):
    dst = dr_in / f'{i:05d}.png'
    if not dst.exists():
        dst.symlink_to(f)
!ffmpeg -y -framerate 25 -i {dr_in}/%05d.png -c:v libx264 -pix_fmt yuv420p {WORK}/in.mp4 2>&1 | tail -3

In [ ]:
DR_OUT = OUTPUT_DIR / 'deepremaster'
DR_OUT.mkdir(parents=True, exist_ok=True)
DR_BENCH = BENCH_DIR / 'deepremaster.json'

t0 = time.time()
cmd = [
    'python', '/content/deepremaster/remaster.py',
    '--input', str(WORK / 'in.mp4'),
    '--output', str(WORK / 'out'),
    '--disable_colorization',  # keep B&W; DeepRemaster's colorize is off-target here
]
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd='/content/deepremaster', capture_output=True, text=True)
print(proc.stdout[-2000:])
print('STDERR:', proc.stderr[-2000:])
# Find the output video and split back to frames aligned with original names.
out_mp4 = next((WORK / 'out').rglob('*.mp4'), None)
if out_mp4 is not None:
    !ffmpeg -y -i {out_mp4} -start_number 0 {WORK}/out/%05d.png 2>&1 | tail -3
    for i, f in enumerate(frames):
        src = WORK / 'out' / f'{i:05d}.png'
        if src.exists():
            shutil.copy(src, DR_OUT / f.name)
wall = time.time() - t0
n_out = len(list(DR_OUT.glob('*.png')))
DR_BENCH.write_text(json.dumps({
    'model': 'deepremaster',
    'wall_s': round(wall, 1),
    'n_frames': n_out,
    'fps': round(n_out / max(wall, 1e-6), 2),
    'returncode': proc.returncode,
    'cmd': cmd,
}, indent=2))
print(f'DeepRemaster: {n_out} frames, {wall:.1f}s')

---
## 4. Done

Outputs are on Drive under `s06-eval/output/{bofbl,rrtn,deepremaster}/`.
Benchmarks under `s06-eval/bench/`.

Pull them back locally and run `colab/s06_eval/build_comparison.py` to render `runs/s06-eval/comparison.html`.